In [ ]:
# script for analysing the ussage if POS, feats and deprels in the UD data

In [ ]:
import pandas as pd
from os.path import join

In [ ]:
PATH = 'conllu_files'
LANGUAGE = 'Ruuli' # can be changed
FILE_NAME = '_training_file.conllu' # can be changed
FILE_PATH = join(PATH, LANGUAGE + FILE_NAME)

In [ ]:
!.venv/bin/python conllu_from_xlsx.py Ruuli

## Preprocessing

In [ ]:
with open(FILE_PATH, 'r') as f:
    lines = f.read().split('\n')
    lines = [line.split('\t') for line in lines if line != '']
lines

In [ ]:
col_names = ['SENTID', 'ID', 'FORM', 'LEMMA', 'UPOSTAG', 'XPOSTAG', 'FEATS', 'HEAD', 'DEPREL', 'DEPS', 'MISC']

In [ ]:
sent_id = ''
conllu = []
for line in lines:
    if line[0].startswith('# sent_id ='):
        sent_id = line[0].split(' = ')[1]
    elif line[0][0] != '#':
        conllu_dict = dict()
        conllu_dict[col_names[0]] = sent_id
        for i, name in enumerate(col_names[1:]):
            conllu_dict[name] = line[i]
        conllu.append(conllu_dict)
df = pd.DataFrame.from_dict(conllu)
df

In [ ]:
def split_feats(feats : str):
    # print(feats)
    if '_' in feats:
        return '_'
    else:
        print(feats)
        return [(feat.split('=')[0], feat.split('=')[1]) for feat in feats.split('|')]

In [ ]:
df['FEATS'] = df['FEATS'].apply(lambda x: split_feats(x))
df

## Analysis

### POS

In [ ]:
all_pos = sorted(list(set(df['UPOSTAG'])))
all_pos = [pos for pos in all_pos if pos != '_']
all_pos
# df['UPOSTAG'].value_counts()

In [ ]:
# words that have specified pos
# closed class words
# ['ADP', 'AUX', 'CCONJ', 'DET', 'NUM', 'PART', 'PRON', 'SCONJ']
lemmas = df[df['UPOSTAG'] == 'AUX']['LEMMA'].unique()
lemmas.sort()
lemmas

In [ ]:
# finding specific ocurences
df[df['UPOSTAG'] == 'PRON'][df['LEMMA'] == 'yo']['SENTID']

### FEATS

In [ ]:
def get_all_feats(df_slice : pd.DataFrame):
    all_feats = dict()
    for feats in df_slice['FEATS']:
        if type(feats) == list:
            for feat in feats:
                if feat[0] not in all_feats.keys():
                    all_feats[feat[0]] = [feat[1]]
                elif feat[1] not in all_feats[feat[0]]:
                    all_feats[feat[0]].append(feat[1])
    all_feats = {key: sorted(all_feats[key]) for key in sorted(all_feats)}
    return all_feats

In [ ]:
# all feats and its vlues:
all_feats = get_all_feats(df)
feat_names = list(all_feats.keys())
feat_names.sort()
feat_names

In [ ]:
# feats used only once
for feat in feat_names:
    if len(df[df['FEATS'].apply(lambda x: any(fv[0] == feat for fv in x))]) == 1:
        print(feat, '\t', df[df['FEATS'].apply(lambda x: any(fv[0] == feat for fv in x))]['SENTID'].values[0])

In [ ]:
# pos by feats
pos_by_feats = dict()
for feat in all_feats.keys():
    pos_by_feats[feat] = []
    for i in df.index:
        if df['UPOSTAG'].iloc[i] not in pos_by_feats[feat] and df['FEATS'].iloc[i] != '_':
            word_feats = [feat[0] for feat in df['FEATS'].iloc[i]]
            if feat in word_feats:
                pos_by_feats[feat].append(df['UPOSTAG'].iloc[i])
    pos_by_feats[feat].sort()
for feat in feat_names:
    print(feat, '\n\t(`' + '`, `'.join(all_feats[feat]) + '`)',
          '\n\t', ', '.join(pos_by_feats[feat]))

In [ ]:
# pos and values by feats
pos_value_by_feats = {feat: {pos: set() for pos in all_pos} for feat in feat_names}
for i in df.index:
    if df['FEATS'].iloc[i] != '_':
        for fv in df['FEATS'].iloc[i]:
            pos_value_by_feats[fv[0]][df['UPOSTAG'].iloc[i]].add(fv[1])

max_len_value = max([len(value) for values in all_feats.values() for value in values])

print('\t' + ' '*(max_len_value + 1) + ' '.join(all_pos))
for feat in pos_value_by_feats.keys():
    print(feat)
    for value in all_feats[feat]:
        print(f'\t{value:<{max_len_value}}', end=' ')
        pos_by_value = [pos for pos in pos_value_by_feats[feat].keys() if value in pos_value_by_feats[feat][pos]]
        print(' '.join([pos if pos in pos_by_value else '_'*len(pos) for pos in all_pos]))

In [ ]:
# feats by pos
for pos in sorted(df['UPOSTAG'].unique()):
    print(pos)
    print(get_all_feats(df[df['UPOSTAG']==pos]))
    print()

In [ ]:
def check_inst(pos : str, feat : str, value : str):
    print(pos, feat, value)
    for i, feats in enumerate(df[df['UPOSTAG']==pos]['FEATS']):
        if type(feats) == list:
            if (feat, value) in feats or (feat in [fv[0] for fv in feats] and value == '*'):
                print(df[df['UPOSTAG']==pos].iloc[i][['SENTID', 'FORM']])
    print()

In [ ]:
# checking some instances, '*' for any feature value
# POS, feature, feature value
to_check = [['INTJ', 'Polarity', 'Neg']]
for inst in to_check:
    check_inst(inst[0], inst[1], inst[2])

In [ ]:
def find_lemmas(pos : str, feat : str, value : str):
    print(pos, feat, value)
    lemmas_by_value = dict()
    for i, feats in enumerate(df[df['UPOSTAG']==pos]['FEATS']):
        if type(feats) == list:
            if (feat, value) in feats or (feat in [fv[0] for fv in feats] and value == '*'):
                value_found = [fv[1] for fv in feats if fv[0] == feat][0]
                if value_found not in lemmas_by_value.keys():
                    lemmas_by_value[value_found] = set()
                lemmas_by_value[value_found].add(df[df['UPOSTAG']==pos].iloc[i]['LEMMA'])
    common_lemmas = set()
    for value in lemmas_by_value.keys():
        common_lemmas = common_lemmas.intersection(lemmas_by_value[value])
        print(f'\t{value}: {", ".join(sorted(lemmas_by_value[value]))}')

    if len(lemmas_by_value.keys()) > 0:
        print(f'Common lemmas for all values: {", ".join(sorted(common_lemmas))}' if len(common_lemmas) > 0 else 'No common lemmas for all values')

In [ ]:
# finding lemmas with specified pos, feat and value, '*' for any feature value
# POS, feature, feature value
lemmas_to_find = [['INTJ', 'Polarity', 'Pos']]
for inst in lemmas_to_find:
    find_lemmas(inst[0], inst[1], inst[2])

### DEPREL

In [ ]:
df['DEPREL'].value_counts()

In [ ]:
deprel = df['DEPREL'].unique()
deprel.sort()
deprel

In [ ]:
all_deprel = list(df['DEPREL'].unique())
sorted([deprel for deprel in all_deprel if ':' in deprel])